# 12 — Resource analysis: when does the amortised posterior pay off? (System I, N3)

The old notebook compared SBI with NUTS. In the N3 re-run there is no valid NUTS baseline (the old likelihood is invalid, plan SI-1; decision 2026-11-09), so the comparison is with the **classical estimators that were built and validated: the EKF and the UKF** (notebook 16), with the cost of NUTS quoted from the earlier implementation as
context only (a cost of the sampler, not of a validated baseline).

**Measurements (fixed before running; runbook §2f row g-12; a quiet machine: the notebook stops if the 1-minute load average exceeds 3).** (1) SBI inference per window with the production ensemble (5 members, notebook 36): feature computation plus 1,000 draws, a single window (median of 100 calls) and a batch of 300 windows; also 100 and 2,000 draws. (2) EKF and UKF (augmented, tuned as notebook 16) per window, 50 windows each, measured here.
(3) Training cost of the production posterior: simulating 2,000 windows (scaled ×10 to the 20,000-window bank) and the training times stored with the five members. (4) Break-even N* = T_train / (t_filter − t_SBI) windows, for single-window and batched SBI; and the cost of a **retraining** after a plant change (the amortised posterior must be retrained, a filter need not).
**Checks:** finite; load average below 3; the SBI single-window median within 40% of notebook 10's 53 ms. No decision rule.


In [1]:
import os, sys, time, pickle, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "system_I" and NB_DIR.parent.name == "notebooks", f"run from notebooks/system_I (cwd {NB_DIR})"
ROOT = NB_DIR.parent.parent
for p in (ROOT / "src", ROOT / "scripts"):
    sys.path.insert(0, str(p))
os.environ["CSTR_SENSOR_MODEL"] = "N3"
warnings.filterwarnings("ignore")
load0 = os.getloadavg()[0]; print("1-minute load average at start:", round(load0, 2))
assert load0 < 3.0, f"machine not quiet (load {load0:.1f}): re-run when nothing else is running"
import importlib, ws_eval, ws_phase3, ws_filters, nb35_measured_inlet, nb36_production_measured_inlet, ws_inlet
for _m in (ws_eval, ws_phase3, ws_filters, nb35_measured_inlet, nb36_production_measured_inlet, ws_inlet):
    importlib.reload(_m)
we, P3, F, M, P36, wi = ws_eval, ws_phase3, ws_filters, nb35_measured_inlet, nb36_production_measured_inlet, ws_inlet
R = ROOT / "results"; FIGS = ROOT / "figures"; FIGS.mkdir(exist_ok=True); T0 = time.time()
d1 = P3.load_d1(); post, prod = P3.load_production_posterior()


1-minute load average at start: 2.46


## 1. SBI inference time per window (production ensemble, 5 members)

In [2]:
def one(i, n):
    f = P36.features_from_raw(d1["x"][i:i + 1], d1["inlet_measured"][i:i + 1]); return we.sample_within_prior(post, f, n, rng_seed=i)
one(0, 1000)
rows = []
for n in (100, 1000, 2000):
    tt = []
    for i in range(100):
        t = time.perf_counter(); one(i, n); tt.append(time.perf_counter() - t)
    rows.append(dict(item=f"single window: features + {n} draws", ms_median=1000 * float(np.median(tt)), ms_p05=1000 * float(np.percentile(tt, 5)), ms_p95=1000 * float(np.percentile(tt, 95))))
t = time.perf_counter(); fb = P36.features_from_raw(d1["x"], d1["inlet_measured"]); t_f = time.perf_counter() - t
t = time.perf_counter(); we.sample_within_prior(post, fb, 1000, rng_seed=1); t_b = time.perf_counter() - t
rows.append(dict(item="batch of 300 windows, 1000 draws: per window", ms_median=1000 * t_b / 300, ms_p05=np.nan, ms_p95=np.nan))
rows.append(dict(item="batch of 300 windows: features per window", ms_median=1000 * t_f / 300, ms_p05=np.nan, ms_p95=np.nan))
sbi = pd.DataFrame(rows); display(sbi.round(3)); sbi.to_csv(R / "12_N3_sbi_timing.csv", index=False)
t_sbi_single = sbi.iloc[1].ms_median / 1000; t_sbi_batch = sbi.iloc[3].ms_median / 1000
assert np.isfinite(sbi.ms_median).all() and abs(t_sbi_single / 0.0529 - 1) < 0.40, ("single-window time differs from notebook 10 by more than 40%", t_sbi_single)


,item,ms_median,ms_p05,ms_p95
0,single window: features + 100 draws,20.946,20.162,40.846
1,single window: features + 1000 draws,51.367,26.527,53.060
2,single window: features + 2000 draws,99.108,97.819,114.430
3,"batch of 300 windows, 1000 draws: per window",13.202,NaN,NaN
4,batch of 300 windows: features per window,0.296,NaN,NaN


## 2. EKF and UKF per window (augmented, as notebook 16; 50 windows)

In [3]:
ids = np.arange(0, 300, 6)[:50]; ft = {}
for method in ("ekf", "ukf"):
    for qs, rs, lab in ((1.0, 1.0, "untuned"), ({"ekf": 2.0, "ukf": 0.25}[method], 2.0, "tuned")):
        tt = []
        for i in ids:
            ti, tci = (float(v) for v in d1["inlet_measured"][i])
            t = time.perf_counter(); F.estimate_window(d1["x"][i], method, ti, tci, F.CI0, augment=True, q_scale=qs, r_scale=rs); tt.append(time.perf_counter() - t)
        ft[(method, lab)] = float(np.median(tt))
fil = pd.DataFrame([dict(method=m.upper(), setting=l, ms_per_window=1000 * v) for (m, l), v in ft.items()]); display(fil.round(2)); fil.to_csv(R / "12_N3_filter_timing.csv", index=False)


,method,setting,ms_per_window
0,EKF,untuned,32.45
1,EKF,tuned,32.48
2,UKF,untuned,484.18
3,UKF,tuned,483.72


## 3. Training cost of the production posterior and the break-even

In [4]:
t = time.perf_counter(); th = we.prior_draws(2000, 123); inl = M._inlets(2000, 124); b29 = wi.simulate(th, inl, M.PROTOCOL, 125, features="S0_base29", **M.NOISE); t_sim2k = time.perf_counter() - t
t_bank = t_sim2k * 10                                     # the 20,000-window bank
train_s = [pickle.load(open(m, "rb"))["metadata"]["train_s"] for m in prod["members"]]
T_train = t_bank + float(np.sum(train_s))
print(f"simulating 2,000 windows: {t_sim2k:.1f}s -> 20,000-window bank {t_bank:.0f}s | training of the 5 members (stored): {np.round(train_s).astype(int).tolist()} s, total {sum(train_s):.0f}s | T_train = {T_train:.0f}s = {T_train / 60:.1f} min (this machine, 10 cores, torch 4 threads)")
rows = []
for m in ("EKF", "UKF"):
    tf = fil[(fil.method == m) & (fil.setting == "tuned")].ms_per_window.iloc[0] / 1000
    for mode, ts in (("batched SBI", t_sbi_batch), ("single-window SBI", t_sbi_single)):
        rows.append(dict(filter=m, sbi_mode=mode, t_filter_ms=1000 * tf, t_sbi_ms=1000 * ts, break_even_windows=(T_train / (tf - ts)) if tf > ts else np.inf))
be = pd.DataFrame(rows); display(be.round(1)); be.to_csv(R / "12_N3_break_even.csv", index=False)
print("break-even N* = T_train / (t_filter - t_SBI); infinite where the SBI window is slower than the filter window.")
print("a retraining after a plant change costs another T_train = %.1f min (simulation + 5 trainings); a filter needs none (but its model constants must be right)." % (T_train / 60))
print(f"notebook wall time {(time.time() - T0) / 60:.1f} min | 1-minute load average at the end: {os.getloadavg()[0]:.2f}")


simulating 2,000 windows: 9.4s -> 20,000-window bank 94s | training of the 5 members (stored): [52, 55, 55, 57, 47] s, total 266s | T_train = 360s = 6.0 min (this machine, 10 cores, torch 4 threads)


,filter,sbi_mode,t_filter_ms,t_sbi_ms,break_even_windows
0,EKF,batched SBI,32.5,13.2,18680.6
1,EKF,single-window SBI,32.5,51.4,inf
2,UKF,batched SBI,483.7,13.2,765.5
3,UKF,single-window SBI,483.7,51.4,833.1


break-even N* = T_train / (t_filter - t_SBI); infinite where the SBI window is slower than the filter window.
a retraining after a plant change costs another T_train = 6.0 min (simulation + 5 trainings); a filter needs none (but its model constants must be right).
notebook wall time 1.4 min | 1-minute load average at the end: 2.10


## 7. Interpretation

Measured on a machine with a 1-minute load average of 2.5 at the start (below the stop level of 3); 10 cores, no GPU.

**Per-window cost.** The production posterior (5-member ensemble) takes **51 ms for one window** (features plus 1,000 draws; 21 ms for 100 draws, 99 ms for 2,000) and **13 ms per window in a batch of 300** (plus 0.3 ms for the features). The EKF takes 32 ms per window and the UKF **484 ms**, the same whether untuned or tuned. A single SBI window therefore costs about as much as an EKF window (1.6 times) and a tenth of a UKF window; in a batch SBI is 2.5 times cheaper than the EKF and 37 times cheaper than the UKF.
(Notebook 10 measured 53 ms and 9.8 ms; the batch figure here is higher because the machine was less quiet.)

**Training cost.** A production posterior costs 94 s of simulation (the 20,000-window bank, extrapolated from 2,000 windows) and 266 s of training (5 members at 47–57 s) = **6.0 minutes**; a **retraining after a plant change costs the same again**. A filter needs no training but needs its model constants right; the amortised posterior needs the simulator right and a retraining when the plant changes.

**Break-even** (N* = T_train / (t_filter − t_SBI)): against the UKF, **766 windows** with batched SBI and 833 with single-window calls (about a month of hourly windows); against the EKF, **18,700 windows** (about two years of hourly windows) with batched SBI, and **never** with single-window calls, because one SBI window is slower than one EKF window. Where the decision depends on cost alone the amortised posterior is attractive against the UKF and marginal against the EKF.
It is not decided by cost, though: the notebook-16 comparison shows that the filters are less well calibrated and fail without warning in a regime a plant visits routinely (valve at its lower limit), while the posterior is calibrated and at the information bound (notebooks 08, 16). The cost argument is a tie-breaker, not the case.

**NUTS is not compared** (decision C): its cost is a property of an earlier, invalid implementation. For context only, the earlier runs recorded in the repository measured about 300 s per window with a heavy tail up to about an hour (`scripts/nb20_nuts_worker.py` docstring, HANDOFF.md); the Gaussian-summary likelihood behind them is invalid (plan SI-1), so the figure says what the sampler costs, not what a valid baseline would.

**Ledger.** Supports: SBI per-window cost comparable to an EKF and well below a UKF; break-even 770–830 windows against the UKF; training 6 minutes. Documents: no break-even against the EKF for single-window use; retraining as the price of amortisation.